# SAC Segway Balancer
Soft Actor-Critic directly controlling the `drive_motor` (horizontal force) to balance the inverted-pendulum segway.

**Model:** `segway_ed.xml`  
**State:** `[x, ẋ, θ, θ̇]` (4-dim)  
**Action:** force ∈ [-100, 100] N on x_slide (1-dim continuous)  
**Goal:** keep θ ≈ 0 (upright) and x ≈ 0 (centred)

In [1]:
# ── Cell 1 · Imports & Config ─────────────────────────────────────────────────
import mujoco
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os, json, imageio, base64
from collections import deque
from IPython.display import display, HTML
from PIL import Image as PILImage, ImageDraw

# ── paths ──────────────────────────────────────────────────────────────────────
XML_PATH = "segway_ed.xml"   # ← update if needed
SAVE_DIR = "SAC_Saves"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── environment ────────────────────────────────────────────────────────────────
X_LIMIT       = 4.5          # walls are at ±5 m
THETA_LIMIT   = np.pi / 2    # 90 ° — fall threshold
MAX_STEPS     = 500         # max steps per episode (2 s at dt=0.002)

# ── SAC hyperparameters ────────────────────────────────────────────────────────
HIDDEN_DIM    = 256
LR_ACTOR      = 3e-4
LR_CRITIC     = 3e-4
LR_ALPHA      = 3e-4
GAMMA         = 0.99
TAU           = 0.005        # soft-update coefficient
BUFFER_SIZE   = 500_000
BATCH_SIZE    = 256
START_STEPS   = 5_000        # random actions before first update
UPDATE_AFTER  = 1_000        # start updating after this many steps
UPDATE_EVERY  = 50           # update every N env steps
UPDATES_PER   = 1           # gradient steps per update call
LOG_EVERY     = 50           # episodes between console prints

# ── reproducibility ────────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"MuJoCo version: {mujoco.__version__}")

Device: cpu
MuJoCo version: 3.8.0


In [2]:
# ── Cell 2 · Segway Environment ───────────────────────────────────────────────

class SegwayEnv:
    """
    Thin MuJoCo wrapper for segway_ed.xml.

    Joints (in order):
        qpos[0] / qvel[0]  →  x_slide   (horizontal position / velocity)
        qpos[1] / qvel[1]  →  body_pitch (tilt angle / angular rate)

    Actuator:
        ctrl[0]  →  drive_motor on x_slide,  range [-100, 100] N

    Observation  (4-dim, normalised):
        [x / X_LIMIT,  ẋ / 5,  θ / π,  θ̇ / 10]

    Curriculum:
        self.difficulty  →  max starting tilt in radians.
        Start at 0.05 rad (≈3°) and increase during training.
    """

    OBS_DIM = 4
    ACT_DIM = 1
    ACT_MAX = 100.0            # raw actuator range

    # normalisation denominators
    _OBS_SCALE = np.array([X_LIMIT, 5.0, np.pi, 10.0], dtype=np.float32)

    def __init__(self, xml_path=XML_PATH):
        self.model      = mujoco.MjModel.from_xml_path(xml_path)
        self.data       = mujoco.MjData(self.model)
        self.dt         = self.model.opt.timestep
        self.difficulty = 0.05   # grows via set_difficulty()

        self._renderer  = None   # lazy — only created when render() is called

    # ── public API ──────────────────────────────────────────────────────────────

    def reset(self, tilt=None, x=None):
        """Reset to a random (or fixed) starting state."""
        mujoco.mj_resetData(self.model, self.data)

        d = self.difficulty
        t = tilt if tilt is not None else np.random.uniform(-d, d)
        p = x    if x    is not None else np.random.uniform(-0.3, 0.3)

        self.data.qpos[0] = p
        self.data.qpos[1] = t
        self.data.qvel[:] = np.zeros(self.model.nv)
        mujoco.mj_forward(self.model, self.data)
        return self._obs()

    def step(self, action_normalised):
        """
        action_normalised: float in [-1, 1]  (SAC tanh output)
        Returns: obs, reward, terminated, info
        """
        force = float(np.clip(action_normalised, -1.0, 1.0)) * self.ACT_MAX
        self.data.ctrl[0] = force
        mujoco.mj_step(self.model, self.data)

        obs        = self._obs()
        raw        = self._raw()
        terminated = bool(abs(raw[0]) > X_LIMIT or
                          abs(raw[2]) > THETA_LIMIT)
        reward     = self._reward(raw, action_normalised, terminated)
        return obs, reward, terminated, {"raw": raw}

    def render(self, height=480, width=640):
        if self._renderer is None:
            self._renderer = mujoco.Renderer(
                self.model, height=height, width=width)
        self._renderer.update_scene(self.data)
        return self._renderer.render()

    def set_difficulty(self, d):
        self.difficulty = float(np.clip(d, 0.0, THETA_LIMIT))

    def close(self):
        pass

    # ── internals ───────────────────────────────────────────────────────────────

    def _raw(self):
        """Unnormalised state: [x, ẋ, θ, θ̇]"""
        return np.array([
            self.data.qpos[0],   # x
            self.data.qvel[0],   # ẋ
            self.data.qpos[1],   # θ  (body_pitch)
            self.data.qvel[1],   # θ̇
        ], dtype=np.float32)

    def _obs(self):
        """Normalised observation clipped to [-1, 1]."""
        return np.clip(self._raw() / self._OBS_SCALE, -1.0, 1.0)

    def _reward(self, raw, action_norm, terminated):
        """
        Dense reward design:
          + upright bonus   — peaks at 1.0 when θ=0, falls off as |θ| grows
          − position cost   — penalise drifting from centre
          − action cost     — small penalty for large forces (energy efficiency)
          − termination     — large negative on fall / out-of-bounds
        """
        x, _, theta, _ = raw

        if terminated:
            return -20.0

        upright  =  np.cos(theta)                    # 1 at θ=0, −1 at θ=π
        pos_cost = -0.1 * (x / X_LIMIT) ** 2
        act_cost = -0.001 * action_norm ** 2

        return upright + pos_cost + act_cost


# ── quick sanity check ─────────────────────────────────────────────────────────
env = SegwayEnv(XML_PATH)
obs = env.reset()
print(f"obs shape : {obs.shape}")
print(f"obs reset : {obs}")
obs2, r, done, info = env.step(0.0)
print(f"obs step  : {obs2}")
print(f"reward    : {r:.4f}  |  terminated: {done}")
print("SegwayEnv ✅")

obs shape : (4,)
obs reset : [ 0.06009524  0.         -0.00399351  0.        ]
obs step  : [ 6.0095236e-02 -4.6982836e-06 -3.9935936e-03 -2.5558471e-05]
reward    : 0.9996  |  terminated: False
SegwayEnv ✅


In [3]:
# ── Cell 3 · Replay Buffer ────────────────────────────────────────────────────

class ReplayBuffer:
    """
    Simple numpy-backed circular buffer.
    Stores (s, a, r, s', done) transitions.
    """

    def __init__(self, obs_dim, act_dim, capacity=BUFFER_SIZE):
        self.capacity = capacity
        self.ptr      = 0
        self.size     = 0

        self.obs  = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.act  = np.zeros((capacity, act_dim), dtype=np.float32)
        self.rew  = np.zeros((capacity, 1),       dtype=np.float32)
        self.obs2 = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.done = np.zeros((capacity, 1),       dtype=np.float32)

    def push(self, obs, act, rew, obs2, done):
        self.obs [self.ptr] = obs
        self.act [self.ptr] = act
        self.rew [self.ptr] = rew
        self.obs2[self.ptr] = obs2
        self.done[self.ptr] = float(done)
        self.ptr  = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size=BATCH_SIZE):
        idx = np.random.randint(0, self.size, size=batch_size)
        return (
            torch.FloatTensor(self.obs [idx]).to(DEVICE),
            torch.FloatTensor(self.act [idx]).to(DEVICE),
            torch.FloatTensor(self.rew [idx]).to(DEVICE),
            torch.FloatTensor(self.obs2[idx]).to(DEVICE),
            torch.FloatTensor(self.done[idx]).to(DEVICE),
        )

    def __len__(self):
        return self.size


print("ReplayBuffer ✅")

ReplayBuffer ✅


In [4]:
# ── Cell 4 · SAC Networks ─────────────────────────────────────────────────────

LOG_STD_MIN = -5
LOG_STD_MAX =  2


def mlp(in_dim, out_dim, hidden=HIDDEN_DIM):
    """Two-hidden-layer MLP with ReLU activations."""
    return nn.Sequential(
        nn.Linear(in_dim, hidden), nn.ReLU(),
        nn.Linear(hidden, hidden), nn.ReLU(),
        nn.Linear(hidden, out_dim),
    )


class Actor(nn.Module):
    """
    Squashed Gaussian policy.
    Outputs mean and log_std, samples via reparameterisation,
    applies tanh → action in (-1, 1).
    """

    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net     = mlp(obs_dim, HIDDEN_DIM)
        self.mu_head = nn.Linear(HIDDEN_DIM, act_dim)
        self.ls_head = nn.Linear(HIDDEN_DIM, act_dim)

    def forward(self, obs):
        h       = self.net(obs)
        mu      = self.mu_head(h)
        log_std = self.ls_head(h).clamp(LOG_STD_MIN, LOG_STD_MAX)
        return mu, log_std

    def sample(self, obs):
        """Returns (action_tanh, log_prob)."""
        mu, log_std = self(obs)
        std         = log_std.exp()
        dist        = torch.distributions.Normal(mu, std)
        x           = dist.rsample()              # reparameterised
        action      = torch.tanh(x)

        # log prob with tanh squashing correction
        log_prob    = dist.log_prob(x) - torch.log(
                         1 - action.pow(2) + 1e-6)
        log_prob    = log_prob.sum(-1, keepdim=True)
        return action, log_prob

    def deterministic(self, obs):
        """Greedy action (mean of distribution, tanh-squashed)."""
        mu, _ = self(obs)
        return torch.tanh(mu)


class Critic(nn.Module):
    """
    Twin Q-networks (clipped double-Q trick).
    Input: (obs, action)  →  Q1, Q2
    """

    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.q1 = mlp(obs_dim + act_dim, 1)
        self.q2 = mlp(obs_dim + act_dim, 1)

    def forward(self, obs, act):
        x  = torch.cat([obs, act], dim=-1)
        return self.q1(x), self.q2(x)

    def q_min(self, obs, act):
        q1, q2 = self(obs, act)
        return torch.min(q1, q2)


print("SAC networks defined ✅")

SAC networks defined ✅


In [5]:
# ── Cell 5 · SAC Agent ────────────────────────────────────────────────────────

import copy

class SACAgent:
    """
    Full SAC with automatic entropy tuning.

    References:
      Haarnoja et al. 2018 — Soft Actor-Critic
      Haarnoja et al. 2019 — SAC with auto alpha
    """

    def __init__(self, obs_dim, act_dim):
        # ── networks ────────────────────────────────────────────────────────────
        self.actor        = Actor(obs_dim, act_dim).to(DEVICE)
        self.critic       = Critic(obs_dim, act_dim).to(DEVICE)
        self.critic_targ  = copy.deepcopy(self.critic).to(DEVICE)

        # freeze target — updated only via soft copy
        for p in self.critic_targ.parameters():
            p.requires_grad = False

        # ── automatic entropy coefficient ────────────────────────────────────────
        # target entropy = −|A|  (heuristic from Haarnoja 2019)
        self.target_entropy = -act_dim
        self.log_alpha      = torch.zeros(1, requires_grad=True,
                                          device=DEVICE)

        # ── optimisers ───────────────────────────────────────────────────────────
        self.opt_actor  = optim.Adam(self.actor.parameters(),  lr=LR_ACTOR)
        self.opt_critic = optim.Adam(self.critic.parameters(), lr=LR_CRITIC)
        self.opt_alpha  = optim.Adam([self.log_alpha],         lr=LR_ALPHA)

        # ── tracking ─────────────────────────────────────────────────────────────
        self.total_updates = 0

    # ── action selection ────────────────────────────────────────────────────────

    @property
    def alpha(self):
        return self.log_alpha.exp().item()

    @torch.no_grad()
    def act(self, obs, deterministic=False):
        """Single observation → scalar action in [-1, 1]."""
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(DEVICE)
        if deterministic:
            a = self.actor.deterministic(obs_t)
        else:
            a, _ = self.actor.sample(obs_t)
        return a.squeeze(0).cpu().numpy()

    # ── gradient update ─────────────────────────────────────────────────────────

    def update(self, buffer, n_updates=1):
        info = {"critic_loss": 0.0,
                "actor_loss":  0.0,
                "alpha_loss":  0.0,
                "alpha":       self.alpha}

        for _ in range(n_updates):
            obs, act, rew, obs2, done = buffer.sample(BATCH_SIZE)

            # ── critic update ────────────────────────────────────────────────────
            with torch.no_grad():
                a2, lp2  = self.actor.sample(obs2)
                q1_t, q2_t = self.critic_targ(obs2, a2)
                q_targ   = torch.min(q1_t, q2_t) - self.alpha * lp2
                y        = rew + GAMMA * (1 - done) * q_targ

            q1, q2       = self.critic(obs, act)
            critic_loss  = F.mse_loss(q1, y) + F.mse_loss(q2, y)

            self.opt_critic.zero_grad()
            critic_loss.backward()
            nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)
            self.opt_critic.step()

            # ── actor update ─────────────────────────────────────────────────────
            a_new, lp_new = self.actor.sample(obs)
            q_new         = self.critic.q_min(obs, a_new)
            actor_loss    = (self.alpha * lp_new - q_new).mean()

            self.opt_actor.zero_grad()
            actor_loss.backward()
            nn.utils.clip_grad_norm_(self.actor.parameters(), 1.0)
            self.opt_actor.step()

            # ── alpha (entropy) update ───────────────────────────────────────────
            alpha_loss = -(self.log_alpha *
                           (lp_new.detach() + self.target_entropy)).mean()

            self.opt_alpha.zero_grad()
            alpha_loss.backward()
            self.opt_alpha.step()

            # ── soft update of target critic ─────────────────────────────────────
            self._soft_update()

            self.total_updates += 1
            info["critic_loss"] += critic_loss.item()
            info["actor_loss"]  += actor_loss.item()
            info["alpha_loss"]  += alpha_loss.item()

        for k in ("critic_loss", "actor_loss", "alpha_loss"):
            info[k] /= max(n_updates, 1)
        info["alpha"] = self.alpha
        return info

    def _soft_update(self):
        for p, pt in zip(self.critic.parameters(),
                         self.critic_targ.parameters()):
            pt.data.mul_(1 - TAU)
            pt.data.add_(TAU * p.data)

    # ── save / load ─────────────────────────────────────────────────────────────

    def save(self, path):
        torch.save({
            "actor":       self.actor.state_dict(),
            "critic":      self.critic.state_dict(),
            "critic_targ": self.critic_targ.state_dict(),
            "log_alpha":   self.log_alpha.detach().cpu(),
        }, path)
        print(f"  Saved → {path}")

    def load(self, path):
        ck = torch.load(path, map_location=DEVICE)
        self.actor.load_state_dict(ck["actor"])
        self.critic.load_state_dict(ck["critic"])
        self.critic_targ.load_state_dict(ck["critic_targ"])
        self.log_alpha.data.copy_(ck["log_alpha"].to(DEVICE))
        print(f"  Loaded ← {path}")


# ── quick shape check ──────────────────────────────────────────────────────────
agent = SACAgent(SegwayEnv.OBS_DIM, SegwayEnv.ACT_DIM)
dummy_obs = np.zeros(SegwayEnv.OBS_DIM, dtype=np.float32)
dummy_act = agent.act(dummy_obs)
print(f"Actor output shape: {dummy_act.shape}  value: {dummy_act}")
print(f"Alpha (initial): {agent.alpha:.4f}")
print("SACAgent ✅")

Actor output shape: (1,)  value: [0.793513]
Alpha (initial): 1.0000
SACAgent ✅


In [6]:
# ── Cell 6 · Curriculum & Helpers ────────────────────────────────────────────

def update_curriculum(env, ep_rewards, current_diff):
    """
    Gradually increase starting tilt difficulty.
    Only advances when the agent has shown it can handle the current level.
    """
    if len(ep_rewards) < 50:
        return current_diff

    avg = np.mean(ep_rewards[-50:])

    # (avg_reward threshold, difficulty_ceiling) pairs
    schedule = [
        (350, 0.10),   # ±5.7°
        (400, 0.20),   # ±11.5°
        (425, 0.30),   # ±17.2°
        (440, 0.40),   # ±22.9°
        (450, 0.52),   # ±29.8°
    ]

    new_diff = current_diff
    for threshold, target_diff in schedule:
        if avg >= threshold and current_diff < target_diff:
            new_diff = target_diff
            break

    if new_diff != current_diff:
        env.set_difficulty(new_diff)
        deg = np.degrees(new_diff)
        print(f"  📈 Curriculum → difficulty={new_diff:.2f} rad (±{deg:.1f}°)")

    return new_diff


def evaluate(agent, env, n_eval=10):
    """Run n_eval deterministic episodes and return mean reward and success rate."""
    rewards   = []
    successes = []

    for _ in range(n_eval):
        obs  = env.reset()
        ep_r = 0.0
        for _ in range(MAX_STEPS):
            act          = agent.act(obs, deterministic=True)
            obs, r, done, _ = env.step(act[0])
            ep_r        += r
            if done:
                break
        rewards.append(ep_r)
        # success = ran full episode without falling
        successes.append(int(not done))

    return np.mean(rewards), np.mean(successes) * 100


def show_video(frames, filename="segway_sac.mp4"):
    imageio.mimsave(filename, frames, fps=50)
    b64 = base64.b64encode(open(filename, "rb").read()).decode()
    display(HTML(
        f'<video width="640" controls loop>'
        f'<source src="data:video/mp4;base64,{b64}" '
        f'type="video/mp4"></video>'))


print("Helpers ✅")

Helpers ✅


In [ ]:
# ── Cell 7 · Training Loop ────────────────────────────────────────────────────

def train(max_episodes=3000, warm_start=False, tag="v1"):

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    env    = SegwayEnv(XML_PATH)
    agent  = SACAgent(SegwayEnv.OBS_DIM, SegwayEnv.ACT_DIM)
    buffer = ReplayBuffer(SegwayEnv.OBS_DIM, SegwayEnv.ACT_DIM)

    save_path = f"{SAVE_DIR}/sac_segway_{tag}.pth"
    if warm_start and os.path.exists(save_path):
        agent.load(save_path)

    ep_rewards   = []
    best_avg     = -np.inf
    total_steps  = 0
    difficulty   = env.difficulty

    print(f"Training SAC | max_episodes={max_episodes} | device={DEVICE}")
    print(f"{'Ep':>6} | {'Steps':>8} | {'EpR':>8} | "
          f"{'Avg100':>8} | {'SuccRate':>9} | "
          f"{'Alpha':>7} | {'Diff':>5}")
    print("-" * 70)

    for episode in range(1, max_episodes + 1):
        obs      = env.reset()
        ep_r     = 0.0
        ep_steps = 0

        for _ in range(MAX_STEPS):
            # ── select action ────────────────────────────────────────────────────
            if total_steps < START_STEPS:
                # pure random exploration at the start
                raw_act = np.random.uniform(-1.0, 1.0, size=(1,))
            else:
                raw_act = agent.act(obs, deterministic=False)

            # ── step environment ─────────────────────────────────────────────────
            obs2, r, done, _ = env.step(raw_act[0])
            buffer.push(obs, raw_act, r, obs2,
                        float(done))

            obs         = obs2
            ep_r       += r
            total_steps += 1
            ep_steps   += 1

            # ── update ───────────────────────────────────────────────────────────
            if (total_steps >= UPDATE_AFTER and
                    total_steps % UPDATE_EVERY == 0 and
                    len(buffer) >= BATCH_SIZE):
                info = agent.update(buffer, n_updates=UPDATES_PER)

            if done:
                break

        ep_rewards.append(ep_r)

        # ── curriculum ───────────────────────────────────────────────────────────
        difficulty = update_curriculum(env, ep_rewards, difficulty)

        # ── logging ──────────────────────────────────────────────────────────────
        if episode % LOG_EVERY == 0 or episode == 1:
            avg100 = np.mean(ep_rewards[-100:])
            eval_r, sr = evaluate(agent, SegwayEnv(XML_PATH), n_eval=5)

            icon = "✅" if sr >= 80 else "⚠️" if sr >= 40 else "❌"
            print(f"{episode:>6} | {total_steps:>8} | {ep_r:>8.1f} | "
                  f"{avg100:>8.1f} | {sr:>7.0f}% {icon} | "
                  f"{agent.alpha:>7.4f} | {difficulty:.2f}")

            # ── save best policy ─────────────────────────────────────────────────
            if avg100 > best_avg:
                best_avg = avg100
                agent.save(save_path)

    print(f"\n✅ Training done | best avg100={best_avg:.1f}")
    agent.save(f"{SAVE_DIR}/sac_segway_{tag}_final.pth")
    return agent, ep_rewards


# ── run training ──────────────────────────────────────────────────────────────
agent, ep_rewards = train(max_episodes=3000, tag="v1")

Training SAC | max_episodes=3000 | device=cpu
    Ep |    Steps |      EpR |   Avg100 |  SuccRate |   Alpha |  Diff
----------------------------------------------------------------------
     1 |      500 |    496.8 |    496.8 |     100% ✅ |  1.0000 | 0.05
  Saved → SAC_Saves/sac_segway_v1.pth
  📈 Curriculum → difficulty=0.10 rad (±5.7°)
    50 |    25000 |    496.1 |    494.4 |     100% ✅ |  0.8656 | 0.10
  📈 Curriculum → difficulty=0.20 rad (±11.5°)
  📈 Curriculum → difficulty=0.30 rad (±17.2°)
  📈 Curriculum → difficulty=0.40 rad (±22.9°)
  📈 Curriculum → difficulty=0.52 rad (±29.8°)


In [ ]:
import json, os
import numpy as np

# ep_rewards wasn't saved separately — rebuild a dummy list from the checkpoint
# OR just skip the plot and go straight to evaluation
print("ep_rewards not in memory — load the agent and run Cell 10 for evaluation")

ep_rewards not in memory — load the agent and run Cell 10 for evaluation


In [ ]:
import matplotlib
matplotlib.use('Agg')  # ← non-interactive backend, never blocks
import matplotlib.pyplot as plt

rewards = np.array(ep_rewards)
window  = 100
smooth  = np.convolve(rewards, np.ones(window) / window, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(rewards, alpha=0.2, color='steelblue', label='episode reward')
ax.plot(np.arange(window - 1, len(rewards)), smooth,
        color='steelblue', linewidth=2, label=f'{window}-ep moving avg')
ax.axhline(MAX_STEPS * 0.9, color='green', linestyle='--',
           alpha=0.7, label='90% max steps')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('SAC Training — Episode Rewards')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.hist(rewards[-200:], bins=30, color='steelblue', edgecolor='white')
ax.set_xlabel('Episode Reward')
ax.set_ylabel('Count')
ax.set_title('Reward Distribution (last 200 episodes)')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/training_curves.png", dpi=150, bbox_inches='tight')
plt.close()  # ← instead of plt.show()
print("Saved training_curves.png")
from IPython.display import Image
Image(f"{SAVE_DIR}/training_curves.png")

NameError: name 'ep_rewards' is not defined

In [ ]:
# ── Cell 9 · Record Video ─────────────────────────────────────────────────────

def record_video(agent, tag="SAC", tilts=None):
    """
    Record a video testing the agent across a range of starting tilts.
    Overlays: tilt, θ, x, action, step counter.
    """
    if tilts is None:
        tilts = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]

    env    = SegwayEnv(XML_PATH)
    frames = []

    for tilt in tilts:
        obs    = env.reset(tilt=tilt, x=0.0)
        step   = 0
        result = "❌"

        for step in range(MAX_STEPS):
            act           = agent.act(obs, deterministic=True)
            obs, _, done, info = env.step(act[0])
            raw           = info["raw"]

            # render + annotate
            frame = env.render()
            img   = PILImage.fromarray(frame)
            draw  = ImageDraw.Draw(img)
            W, H  = img.size

            # title bar
            draw.rectangle([0, 0, W, 28], fill=(20, 20, 20))
            draw.text((8, 6),
                f"{tag} | start tilt={np.degrees(tilt):.1f}°",
                fill=(255, 255, 255))

            # state readout
            draw.rectangle([0, 30, W, 56], fill=(40, 40, 40))
            draw.text((8, 34),
                f"x={raw[0]:+.3f} m  "
                f"θ={np.degrees(raw[2]):+.1f}°  "
                f"force={act[0]*100:+.1f} N  "
                f"step={step}",
                fill=(180, 220, 255))

            # upright indicator bar
            bx, by, bw = W // 2 - 60, 60, 120
            draw.rectangle([bx, by, bx + bw, by + 14], fill=(60, 60, 60))
            uprightness = max(0.0, 1.0 - abs(raw[2]) / (np.pi / 2))
            bar_color   = (0, 200, 80) if uprightness > 0.8 else \
                          (220, 180, 0) if uprightness > 0.5 else (200, 50, 50)
            draw.rectangle(
                [bx, by, bx + int(bw * uprightness), by + 14],
                fill=bar_color)
            draw.text((bx + bw // 2 - 20, by),
                      f"{uprightness * 100:.0f}% upright",
                      fill=(255, 255, 255))

            frames.append(np.array(img))

            if done:
                break
        else:
            result = "✅"

        # 1-second pause with result
        pause = PILImage.fromarray(
            np.zeros_like(frames[-1]))
        draw_p = ImageDraw.Draw(pause)
        draw_p.text(
            (W // 2 - 100, H // 2 - 10),
            f"start={np.degrees(tilt):.1f}° | "
            f"{result} survived {step} steps",
            fill=(255, 255, 255))
        frames += [np.array(pause)] * 50

        print(f"  θ₀={np.degrees(tilt):>5.1f}° | "
              f"{result} {step}/{MAX_STEPS} steps")

    env.close()
    show_video(frames, f"{SAVE_DIR}/{tag}_test.mp4")
    return frames


# ── load best checkpoint and record ───────────────────────────────────────────
best_agent = SACAgent(SegwayEnv.OBS_DIM, SegwayEnv.ACT_DIM)
best_agent.load(f"{SAVE_DIR}/sac_segway_v1.pth")

frames = record_video(best_agent, tag="SAC_v1")

In [ ]:
# ── Cell 10 · Final Evaluation Summary ────────────────────────────────────────

def full_eval(agent, tilts=None, n_per_tilt=20):
    """
    Sweep over starting tilts and report success rate at each.
    Success = surviving all MAX_STEPS steps without falling.
    """
    if tilts is None:
        tilts = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]

    env = SegwayEnv(XML_PATH)

    print(f"\n{'='*50}")
    print(f"  SAC Final Evaluation")
    print(f"{'='*50}")
    print(f"  {'Tilt':>8} | {'Success%':>9} | {'Avg Reward':>11}")
    print(f"  {'-'*35}")

    for tilt in tilts:
        successes = []
        rewards   = []
        for _ in range(n_per_tilt):
            t   = np.random.uniform(-tilt, tilt)
            obs = env.reset(tilt=t)
            ep_r = 0.0
            fell = False
            for _ in range(MAX_STEPS):
                act = agent.act(obs, deterministic=True)
                obs, r, done, _ = env.step(act[0])
                ep_r += r
                if done:
                    fell = True
                    break
            successes.append(int(not fell))
            rewards.append(ep_r)

        sr   = np.mean(successes) * 100
        avgr = np.mean(rewards)
        icon = "✅" if sr >= 80 else "⚠️" if sr >= 50 else "❌"
        print(f"  {np.degrees(tilt):>6.1f}°  | "
              f"{sr:>8.0f}% {icon} | {avgr:>11.1f}")

    print(f"{'='*50}")
    env.close()


full_eval(best_agent)